**Project Skeleton & Utility Functions**

Our goal in this lesson is not to convert ebooks yet.

Instead, we'll create the foundation that every other lesson will build upon.

# Part I: **Project Skeleton**

## Initial Imports

In [1]:
# ==========================================================
# Universal eBook to PDF Converter v2.0
# ==========================================================

import os
import time
import threading
import subprocess
from pathlib import Path

from google.colab import files

## Application Information

In [2]:
APP_NAME = "Universal eBook to PDF Converter"
VERSION = "2.0"

LINE = "=" * 70

## Global Variables

In [3]:
# Input / Output

input_file = ""
input_path = ""

output_file = ""
output_path = ""


# Conversion Command

command = []


# Progress

progress = 0
conversion_finished = False


# Timing

start_time = None

## Useful Functions

### Header Function

In [4]:
def print_header():

    print("\n" + LINE)
    print(f"{APP_NAME} v{VERSION}".center(70))
    print(LINE)

### Section Function

In [7]:
def print_section(title):

    print("\n")
    print("-" * 70)
    print(title.center(70))
    print("-" * 70)

### Success Message

In [8]:
def success(message):

    print(f"✅ {message}")

In [10]:
# Example
success("Conversion Complete")

✅ Conversion Complete


### Error Message

In [12]:
def error(message):

  print(f"❌ {message}")

### Information Message

In [13]:
def info(message):

    print(f"ℹ️ {message}")

### Warning Message

In [14]:
def warning(message):

    print(f"⚠️ {message}")

### Pause Function

In [15]:
def pause(seconds=1):

    time.sleep(seconds)

### Display Application Banner

In [16]:
print_header()

info("Application Loaded Successfully.")


                Universal eBook to PDF Converter v2.0                 
ℹ️ Application Loaded Successfully.


# Part II: **Configuration Manager**

## Configuration Dictionary

In [17]:
# ==========================================================
# Application Configuration
# ==========================================================

config = {

    # Input / Output
    "input_file": "",
    "input_path": "",
    "output_file": "",
    "output_path": "",

    # PDF Settings
    "paper_size": "a4",

    "margins": {
        "top": 72,
        "bottom": 72,
        "left": 72,
        "right": 72
    },

    "font": {
        "base": 12,
        "pdf_default": 12
    },

    "paragraph": {
        "remove_spacing": False,
        "insert_blank_line": False,
        "blank_line_size": 0.5
    },

    "justify": False,

    # Runtime
    "command": [],

    "progress": 0,

    "conversion_finished": False,

    "start_time": None
}

## Print Configuration

In [18]:
print_section("Current Configuration")

for key, value in config.items():
    print(f"{key:22} : {value}")



----------------------------------------------------------------------
                        Current Configuration                         
----------------------------------------------------------------------
input_file             : 
input_path             : 
output_file            : 
output_path            : 
paper_size             : a4
margins                : {'top': 72, 'bottom': 72, 'left': 72, 'right': 72}
font                   : {'base': 12, 'pdf_default': 12}
paragraph              : {'remove_spacing': False, 'insert_blank_line': False, 'blank_line_size': 0.5}
justify                : False
command                : []
progress               : 0
conversion_finished    : False
start_time             : None


# Part III: **Dependency Manager**

In [54]:
import shutil

In [55]:
def is_calibre_installed():

    return shutil.which("ebook-convert") is not None

In [56]:
def install_calibre():

    print_section("Installing Calibre")

    subprocess.run(
        ["apt-get", "update"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    subprocess.run(
        ["apt-get", "install", "-y", "calibre"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    success("Calibre Installed.")

In [57]:
def setup_dependencies():

    if is_calibre_installed():

        success("Calibre already installed.")

    else:

        install_calibre()

# Part IV: **Interactive User Preference Module**

## Helper Function for Yes/No Questions

In [19]:
def ask_yes_no(question, default=True):
    """
    Ask a Yes/No question and return True or False.
    Pressing Enter selects the default value.
    """

    default_text = "Y/n" if default else "y/N"

    while True:

        choice = input(f"{question} [{default_text}]: ").strip().lower()

        if choice == "":
            return default

        elif choice in ["y", "yes"]:
            return True

        elif choice in ["n", "no"]:
            return False

        else:
            warning("Please enter Y or N.")

## Paper Size Selection

In [20]:
def get_paper_size():

    print_section("Paper Size")

    print("1. A4")
    print("2. Letter")
    print("3. Legal")

    while True:

        choice = input("\nSelect Paper Size [1]: ").strip()

        if choice == "":
            config["paper_size"] = "a4"
            break

        elif choice == "1":
            config["paper_size"] = "a4"
            break

        elif choice == "2":
            config["paper_size"] = "letter"
            break

        elif choice == "3":
            config["paper_size"] = "legal"
            break

        else:
            warning("Invalid choice. Please select 1, 2, or 3.")

    success(f"Paper Size: {config['paper_size'].upper()}")

## Margin Input Helper

In [22]:
def get_margin(name):

    default = config["margins"][name]

    while True:

        value = input(f"{name.capitalize()} Margin [{default}]: ").strip()

        if value == "":
            return default

        try:

            value = int(value)

            if value >= 0:
                return value

            warning("Margin cannot be negative.")

        except ValueError:

            warning("Please enter a valid integer.")

### Margin Settings

In [23]:
def get_margins():

    print_section("Margins")

    config["margins"]["top"] = get_margin("top")
    config["margins"]["bottom"] = get_margin("bottom")
    config["margins"]["left"] = get_margin("left")
    config["margins"]["right"] = get_margin("right")

    success("Margins Updated.")

## Font Settings

In [24]:
def get_font_settings():

    print_section("Font Settings")

    while True:

        value = input(
            f"Base Font Size [{config['font']['base']}]: "
        ).strip()

        if value == "":
            break

        try:

            value = int(value)

            if value > 0:

                config["font"]["base"] = value
                break

            warning("Font size must be positive.")

        except ValueError:

            warning("Please enter a valid integer.")

    success(f"Base Font Size: {config['font']['base']} pt")

## Paragraph Settings

In [25]:
def get_paragraph_settings():

    print_section("Paragraph Settings")

    config["paragraph"]["remove_spacing"] = ask_yes_no(
        "Remove paragraph spacing?",
        default=False
    )

    config["paragraph"]["insert_blank_line"] = ask_yes_no(
        "Insert blank line between paragraphs?",
        default=False
    )

    if config["paragraph"]["insert_blank_line"]:

        while True:

            value = input(
                f"Blank Line Size [{config['paragraph']['blank_line_size']}]: "
            ).strip()

            if value == "":
                break

            try:

                value = float(value)

                if value > 0:

                    config["paragraph"]["blank_line_size"] = value
                    break

                warning("Value must be positive.")

            except ValueError:

                warning("Please enter a valid number.")

    success("Paragraph Settings Updated.")

## Text Justification

In [26]:
def get_justification():

    print_section("Text Alignment")

    config["justify"] = ask_yes_no(
        "Justify text?",
        default=False
    )

    success(
        f"Justification: {'Enabled' if config['justify'] else 'Disabled'}"
    )

## Collect All Preferences

In [27]:
def get_user_preferences():

    print_header()

    info("Configure your PDF output.\n")

    get_paper_size()

    get_margins()

    get_font_settings()

    get_paragraph_settings()

    get_justification()

    print()

    success("All preferences have been collected.")

## Test the Module

In [28]:
get_user_preferences()

print_section("Updated Configuration")

for key, value in config.items():
    print(f"{key:22} : {value}")


                Universal eBook to PDF Converter v2.0                 
ℹ️ Configure your PDF output.



----------------------------------------------------------------------
                              Paper Size                              
----------------------------------------------------------------------
1. A4
2. Letter
3. Legal

Select Paper Size [1]: 1
✅ Paper Size: A4


----------------------------------------------------------------------
                               Margins                                
----------------------------------------------------------------------
Top Margin [72]: 72
Bottom Margin [72]: 72
Left Margin [72]: 72
Right Margin [72]: 72
✅ Margins Updated.


----------------------------------------------------------------------
                            Font Settings                             
----------------------------------------------------------------------
Base Font Size [12]: 16
✅ Base Font Size: 16 pt


-----------------------------

# Part V: **Upload & Validation Module**

## Supported Formats

In [29]:
# ==========================================================
# Supported eBook Formats
# ==========================================================

SUPPORTED_FORMATS = {
    "epub",
    "mobi",
    "azw",
    "azw3",
    "fb2",
    "html",
    "htm",
    "lit",
    "lrf",
    "odt",
    "pdb",
    "pdf",
    "prc",
    "rb",
    "rtf",
    "snb",
    "tcr",
    "txt"
}

## Upload Function

In [30]:
def upload_file():

    print_section("Upload eBook")

    uploaded = files.upload()

    if not uploaded:

        error("No file uploaded.")

        return False

    filename = next(iter(uploaded))

    config["input_file"] = filename

    config["input_path"] = str(Path(filename).resolve())

    success(f"Uploaded: {filename}")

    return True

## Detect Extension

In [31]:
def detect_extension():

    extension = Path(config["input_file"]).suffix.lower()

    extension = extension.replace(".", "")

    return extension

## Validate File

In [32]:
def validate_file():

    print_section("Validating File")

    extension = detect_extension()

    if extension not in SUPPORTED_FORMATS:

        error(f"Unsupported format: .{extension}")

        print()

        print("Supported formats:")

        print(", ".join(sorted(SUPPORTED_FORMATS)))

        return False

    success(f"Detected Format: {extension.upper()}")

    return True

## Generate Output Filename

In [33]:
def prepare_output_file():

    stem = Path(config["input_file"]).stem

    config["output_file"] = f"{stem}.pdf"

    config["output_path"] = str(
        Path(config["output_file"]).resolve()
    )

    info(f"Output File: {config['output_file']}")

## File Summary

In [34]:
def show_file_information():

    print_section("File Information")

    print(f"Input File : {config['input_file']}")

    print(f"Input Path : {config['input_path']}")

    print(f"Output File: {config['output_file']}")

    print(f"Output Path: {config['output_path']}")

## Upload Workflow

In [35]:
def prepare_file():

    if not upload_file():
        return False

    if not validate_file():
        return False

    prepare_output_file()

    show_file_information()

    return True

## Test the Module

In [36]:
print_header()

if prepare_file():

    success("File is ready for conversion.")


                Universal eBook to PDF Converter v2.0                 


----------------------------------------------------------------------
                             Upload eBook                             
----------------------------------------------------------------------


Saving Quantum Machine Learning with Python _ Using Cirq From -- Santanu Pattanayak -- 1st edition, Erscheinungsort nicht ermittelbar, 2021 -- Apress, -- isbn13 9781484265215 -- e0105408f46b206463aebf20234fbb3e -- Anna’s.epub to Quantum Machine Learning with Python _ Using Cirq From -- Santanu Pattanayak -- 1st edition, Erscheinungsort nicht ermittelbar, 2021 -- Apress, -- isbn13 9781484265215 -- e0105408f46b206463aebf20234fbb3e -- Anna’s.epub
✅ Uploaded: Quantum Machine Learning with Python _ Using Cirq From -- Santanu Pattanayak -- 1st edition, Erscheinungsort nicht ermittelbar, 2021 -- Apress, -- isbn13 9781484265215 -- e0105408f46b206463aebf20234fbb3e -- Anna’s.epub


----------------------------------------------------------------------
                           Validating File                            
----------------------------------------------------------------------
✅ Detected Format: EPUB
ℹ️ Output File: Quantum Machine Learning with Python _ Using Cirq From -- Santanu 

# Part VI: **Command Builder Module**

## Base Command

In [58]:
def build_base_command():

    config["command"] = [

        "ebook-convert",

        config["input_path"],

        config["output_path"]

    ]

## Custom Commands

### Paper Size

In [59]:
def add_paper_size():

    config["command"].extend([

        "--paper-size",

        config["paper_size"]

    ])

### Margins

In [60]:
def add_margins():

    margins = config["margins"]

    config["command"].extend([

        "--margin-top",
        str(margins["top"]),

        "--margin-bottom",
        str(margins["bottom"]),

        "--margin-left",
        str(margins["left"]),

        "--margin-right",
        str(margins["right"])

    ])

### Font Settings

In [61]:
def add_font_settings():

    font = config["font"]

    config["command"].extend([

        "--base-font-size",

        str(font["base"])

    ])

### Paragraph Settings

In [62]:
def add_paragraph_settings():

    paragraph = config["paragraph"]

    if paragraph["remove_spacing"]:

        config["command"].append(
            "--remove-paragraph-spacing"
        )

    if paragraph["insert_blank_line"]:

        config["command"].extend([

            "--insert-blank-line",

            "--insert-blank-line-size",

            str(paragraph["blank_line_size"])

        ])

### Justification

In [64]:
def create_custom_css():

    if not config["justify"]:

        return None

    css = """
body{
    text-align:justify;
}
"""

    css_file = "custom_style.css"

    with open(css_file, "w") as file:

        file.write(css)

    return css_file

In [65]:
def add_custom_css():

    css_file = create_custom_css()

    if css_file is None:

        return

    config["command"].extend([

        "--extra-css",

        css_file

    ])

## Display the Command

In [66]:
def show_command():

    print_section("Generated Command")

    print()

    print(" ".join(config["command"]))

## Complete Command

In [67]:
def build_command():

    print_section("Building Command")

    build_base_command()

    add_paper_size()

    add_margins()

    add_font_settings()

    add_paragraph_settings()

    add_custom_css()

    success("Command Generated.")

## Test Everything

In [68]:
build_command()

show_command()



----------------------------------------------------------------------
                           Building Command                           
----------------------------------------------------------------------
✅ Command Generated.


----------------------------------------------------------------------
                          Generated Command                           
----------------------------------------------------------------------

ebook-convert /content/Quantum Machine Learning with Python _ Using Cirq From -- Santanu Pattanayak -- 1st edition, Erscheinungsort nicht ermittelbar, 2021 -- Apress, -- isbn13 9781484265215 -- e0105408f46b206463aebf20234fbb3e -- Anna’s.epub /content/Quantum Machine Learning with Python _ Using Cirq From -- Santanu Pattanayak -- 1st edition, Erscheinungsort nicht ermittelbar, 2021 -- Apress, -- isbn13 9781484265215 -- e0105408f46b206463aebf20234fbb3e -- Anna’s.pdf --paper-size a4 --margin-top 72 --margin-bottom 72 --margin-left 72 --margin-ri

# Part VII: **Conversion Engine**

## Prepare the Environment

In [69]:
def prepare_environment():

    env = os.environ.copy()

    env["QTWEBENGINE_DISABLE_SANDBOX"] = "1"

    return env

## Start the Conversion Timer

In [70]:
def start_timer():

    config["start_time"] = time.time()

## Stop the Timer

In [71]:
def stop_timer():

    elapsed = time.time() - config["start_time"]

    return round(elapsed, 2)

## Execute Conversion

In [72]:
def execute_conversion():

    env = prepare_environment()

    start_timer()

    result = subprocess.run(

        config["command"],

        capture_output=True,

        text=True,

        env=env

    )

    elapsed = stop_timer()

    return result, elapsed

## Handle the Success / Failure

In [73]:
def conversion_success(elapsed):

    success("Conversion completed successfully.")

    print()

    print(f"Time Taken : {elapsed:.2f} seconds")

In [74]:
def conversion_failure(result):

    error("Conversion Failed")

    print()

    print(result.stderr)

## Main Conversion Function

In [75]:
def convert():

    print_section("Converting eBook")

    if not config["command"]:

        error("Conversion command has not been built.")

        return False

    if not os.path.exists(config["input_path"]):

        error("Input file not found.")

        return False

    result, elapsed = execute_conversion()

    if result.returncode == 0:

        conversion_success(elapsed)

        return True

    conversion_failure(result)

    return False

## Test

In [77]:
setup_dependencies()
convert()



----------------------------------------------------------------------
                          Installing Calibre                          
----------------------------------------------------------------------
✅ Calibre Installed.


----------------------------------------------------------------------
                           Converting eBook                           
----------------------------------------------------------------------
✅ Conversion completed successfully.

Time Taken : 84.15 seconds


True

# Part VIII: **Progress Manager**

## Initilize Progress State

In [78]:
import threading

In [79]:
def reset_progress():

    config["progress"] = 0

    config["conversion_finished"] = False

## Define Conversion Stages

In [80]:
PROGRESS_STAGES = [

    "Initializing Converter...",

    "Reading eBook...",

    "Processing Chapters...",

    "Formatting Pages...",

    "Embedding Fonts...",

    "Rendering PDF...",

    "Optimizing Output...",

    "Finalizing PDF..."

]

## Progress Bar Function

In [81]:
def progress_bar():

    stage = 0

    total_stages = len(PROGRESS_STAGES)

    while not config["conversion_finished"]:

        if config["progress"] < 95:

            config["progress"] += 1

        current_stage = min(

            stage,

            total_stages - 1

        )

        elapsed = 0

        if config["start_time"] is not None:

            elapsed = int(time.time() - config["start_time"])

        filled = config["progress"] // 5

        bar = "█" * filled + "░" * (20 - filled)

        print(

            f"\r[{bar}] "

            f"{config['progress']:3d}%   "

            f"{PROGRESS_STAGES[current_stage]}   "

            f"Elapsed: {elapsed}s",

            end=""

        )

        if config["progress"] % 12 == 0:

            stage += 1

        time.sleep(0.20)

    config["progress"] = 100

    bar = "█" * 20

    elapsed = int(time.time() - config["start_time"])

    print(

        f"\r[{bar}] 100%   Conversion Complete!   Elapsed: {elapsed}s"
    )

## Start Progress Manager

In [82]:
def start_progress():

    reset_progress()

    thread = threading.Thread(

        target=progress_bar,

        daemon=True

    )

    thread.start()

    return thread

## Stop Progress Manager

In [83]:
def stop_progress(thread):

    config["conversion_finished"] = True

    thread.join()

## Progress Demonstration

In [84]:
config["start_time"] = time.time()

thread = start_progress()

time.sleep(8)

stop_progress(thread)

[████████████████████] 100%   Conversion Complete!   Elapsed: 8s


## Integrate with the Conversion Engine

In [85]:
def execute_conversion():

    env = prepare_environment()

    start_timer()

    progress_thread = start_progress()

    result = subprocess.run(

        config["command"],

        capture_output=True,

        text=True,

        env=env

    )

    stop_progress(progress_thread)

    elapsed = stop_timer()

    return result, elapsed

# Part IX: **Verification and Conversion Report**

## Verify Output

In [86]:
def verify_output():

    print_section("Verifying Output")

    if os.path.exists(config["output_path"]):

        success("Output PDF verified.")

        return True

    error("Output PDF was not created.")

    return False

## File Size

In [87]:
def get_file_size():

    size = os.path.getsize(config["output_path"])

    size_mb = size / (1024 * 1024)

    return round(size_mb, 2)

## Display Conversion Report

In [88]:
def generate_report():

    print_section("Conversion Report")

    print(f"Input File        : {config['input_file']}")

    print(f"Output File       : {config['output_file']}")

    print(f"Paper Size        : {config['paper_size'].upper()}")

    print()

    print("Margins")

    print("-------")

    print(f"Top               : {config['margins']['top']}")

    print(f"Bottom            : {config['margins']['bottom']}")

    print(f"Left              : {config['margins']['left']}")

    print(f"Right             : {config['margins']['right']}")

    print()

    print("Font")

    print("----")

    print(f"Base Font         : {config['font']['base']} pt")

    print()

    print("Paragraph")

    print("---------")

    print(f"Remove Spacing    : {config['paragraph']['remove_spacing']}")

    print(f"Blank Line        : {config['paragraph']['insert_blank_line']}")

    print(f"Justification     : {config['justify']}")

    print()

    print(f"Output Size       : {get_file_size()} MB")

## Download PDF

In [89]:
def download_pdf():

    print_section("Download")

    files.download(config["output_path"])

    success("Download Started.")

## Finalize Conversion

In [90]:
def finalize_conversion():

    if not verify_output():

        return

    generate_report()

    download_pdf()

## Test

In [91]:
finalize_conversion()



----------------------------------------------------------------------
                           Verifying Output                           
----------------------------------------------------------------------
✅ Output PDF verified.


----------------------------------------------------------------------
                          Conversion Report                           
----------------------------------------------------------------------
Input File        : Quantum Machine Learning with Python _ Using Cirq From -- Santanu Pattanayak -- 1st edition, Erscheinungsort nicht ermittelbar, 2021 -- Apress, -- isbn13 9781484265215 -- e0105408f46b206463aebf20234fbb3e -- Anna’s.epub
Output File       : Quantum Machine Learning with Python _ Using Cirq From -- Santanu Pattanayak -- 1st edition, Erscheinungsort nicht ermittelbar, 2021 -- Apress, -- isbn13 9781484265215 -- e0105408f46b206463aebf20234fbb3e -- Anna’s.pdf
Paper Size        : A4

Margins
-------
Top               : 72
Bottom 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download Started.


# Part X: **Main Controller**

## Main Function

In [92]:
def main():

    print_header()

    # --------------------------------------------------
    # Step 1 : Check Dependencies
    # --------------------------------------------------

    setup_dependencies()

    print()

    # --------------------------------------------------
    # Step 2 : Upload & Validate File
    # --------------------------------------------------

    if not prepare_file():

        error("Application terminated.")

        return

    print()

    # --------------------------------------------------
    # Step 3 : Collect User Preferences
    # --------------------------------------------------

    get_user_preferences()

    print()

    # --------------------------------------------------
    # Step 4 : Build Conversion Command
    # --------------------------------------------------

    build_command()

    print()

    show_command()

    print()

    # --------------------------------------------------
    # Step 5 : Convert
    # --------------------------------------------------

    if not convert():

        error("Conversion unsuccessful.")

        return

    print()

    # --------------------------------------------------
    # Step 6 : Generate Report
    # --------------------------------------------------

    finalize_conversion()

    print()

    success("Application Finished Successfully.")

## Launch the Application

In [94]:
main()


                Universal eBook to PDF Converter v2.0                 
✅ Calibre already installed.



----------------------------------------------------------------------
                             Upload eBook                             
----------------------------------------------------------------------


Saving dokumen.pub_mastering-quantum-computing-with-qiskit-master-the-fundamentals-explore-algorithms-and-build-scalable-quantum-applications-using-ibm-qiskit-to-solve-real-world-problems-english-edition.epub to dokumen.pub_mastering-quantum-computing-with-qiskit-master-the-fundamentals-explore-algorithms-and-build-scalable-quantum-applications-using-ibm-qiskit-to-solve-real-world-problems-english-edition (1).epub
✅ Uploaded: dokumen.pub_mastering-quantum-computing-with-qiskit-master-the-fundamentals-explore-algorithms-and-build-scalable-quantum-applications-using-ibm-qiskit-to-solve-real-world-problems-english-edition (1).epub


----------------------------------------------------------------------
                           Validating File                            
----------------------------------------------------------------------
✅ Detected Format: EPUB
ℹ️ Output File: dokumen.pub_mastering-quantum-computing-with-qiskit-master-the-fundamentals-explore-algorithms-and-build-scal

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download Started.

✅ Application Finished Successfully.
